In [11]:
import pandas as pd
import numpy as np
import joblib
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [12]:
df=sns.load_dataset("titanic")

print("Dataset shape:", df.shape)
print(df.head())
print(df.columns.tolist())

Dataset shape: (891, 15)
   survived  pclass     sex   age  sibsp  parch     fare embarked  class  \
0         0       3    male  22.0      1      0   7.2500        S  Third   
1         1       1  female  38.0      1      0  71.2833        C  First   
2         1       3  female  26.0      0      0   7.9250        S  Third   
3         1       1  female  35.0      1      0  53.1000        S  First   
4         0       3    male  35.0      0      0   8.0500        S  Third   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  
1  woman       False    C    Cherbourg   yes  False  
2  woman       False  NaN  Southampton   yes   True  
3  woman       False    C  Southampton   yes  False  
4    man        True  NaN  Southampton    no   True  
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']


In [13]:
import seaborn as sns

df = sns.load_dataset("titanic")

features = [
    "pclass",
    "sex",
    "age",
    "sibsp",
    "parch",
    "fare",
    "embarked"
]

X = df[features]
y = df["survived"]

In [14]:
print(X.isnull().sum())

pclass        0
sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
dtype: int64


In [19]:
X["age"] = X["age"].fillna(
    X["age"].median()
)

X["fare"] = X["fare"].fillna(
    X["fare"].median()
)

X["embarked"] = X["embarked"].fillna(
    X["embarked"].mode()[0]
)

print(X.isnull().sum())

pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


In [21]:
X["sex"] = X["sex"].map({
    "male": 0,
    "female": 1
})

print(X["sex"].value_counts())

sex
0    577
1    314
Name: count, dtype: int64


In [23]:
X["embarked"] = X["embarked"].map({
    "S": 0,
    "C": 1,
    "Q": 2
})

print(X["embarked"].value_counts())

embarked
0    646
1    168
2     77
Name: count, dtype: int64


In [24]:
print(X.head())
print()
print(X.dtypes)
print()
print("Missing values:")
print(X.isnull().sum())

   pclass  sex   age  sibsp  parch     fare  embarked
0       3    0  22.0      1      0   7.2500         0
1       1    1  38.0      1      0  71.2833         1
2       3    1  26.0      0      0   7.9250         0
3       1    1  35.0      1      0  53.1000         0
4       3    0  35.0      0      0   8.0500         0

pclass        int64
sex           int64
age         float64
sibsp         int64
parch         int64
fare        float64
embarked      int64
dtype: object

Missing values:
pclass      0
sex         0
age         0
sibsp       0
parch       0
fare        0
embarked    0
dtype: int64


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (712, 7)
X_test: (179, 7)
y_train: (712,)
y_test: (179,)


In [26]:
model = DecisionTreeClassifier(
    random_state=42
)

param_grid = {
    "criterion": [
        "gini",
        "entropy"
    ],

    "max_depth": [
        2,
        3,
        4,
        5,
        6,
        7,
        8,
        9,
        10
    ],

    "min_samples_split": [
        2,
        4,
        6,
        8,
        10
    ]
}

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("\nBest Cross-Validation Accuracy:")
print(grid.best_score_)

Best Parameters:
{'criterion': 'gini', 'max_depth': 7, 'min_samples_split': 10}

Best Cross-Validation Accuracy:
0.8217374175120653


In [27]:
best_model = grid.best_estimator_

print("Best Model:")
print(best_model)

Best Model:
DecisionTreeClassifier(max_depth=7, min_samples_split=10, random_state=42)


In [28]:
y_pred = best_model.predict(X_test)

print("Predictions:")
print(y_pred[:20])

Predictions:
[0 0 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 1]


In [29]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

Accuracy : 0.7988826815642458
Precision: 0.8235294117647058
Recall   : 0.6086956521739131
F1 Score : 0.7


In [30]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       0.79      0.92      0.85       110
           1       0.82      0.61      0.70        69

    accuracy                           0.80       179
   macro avg       0.81      0.76      0.77       179
weighted avg       0.80      0.80      0.79       179



In [31]:
cm = confusion_matrix(
    y_test,
    y_pred
)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[101   9]
 [ 27  42]]


In [32]:
sample_passenger = pd.DataFrame(
    [[
        3,      # Pclass
        0,      # Sex: male
        25,     # Age
        0,      # SibSp
        0,      # Parch
        7.25,   # Fare
        0       # Embarked: S
    ]],
    columns=features
)

prediction = best_model.predict(
    sample_passenger
)

print("Prediction:", prediction[0])

if prediction[0] == 1:
    print("Passenger is predicted to survive.")
else:
    print("Passenger is predicted not to survive.")

Prediction: 0
Passenger is predicted not to survive.


In [42]:
import joblib
import os

# Save directly into your Streamlit project's models folder
model_path = r"D:\01_Github repo folders\ML_Portfolio\models\titanic_best.pkl"

joblib.dump(best_model, model_path)

print("Saved successfully!")
print("Path:", model_path)
print("Size:", os.path.getsize(model_path), "bytes")

Saved successfully!
Path: D:\01_Github repo folders\ML_Portfolio\models\titanic_best.pkl
Size: 8057 bytes


In [43]:
import joblib

model = joblib.load(
    r"D:\01_Github repo folders\ML_Portfolio\models\titanic_best.pkl"
)

print("MODEL OK")
print(model)

MODEL OK
DecisionTreeClassifier(max_depth=7, min_samples_split=10, random_state=42)


In [35]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='str')